 <style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

# Knowledge Sources and External Context

## ۱. تعریف‌ها

- **Parametric Knowledge:** دانشی که در وزن‌های مدل ذخیره شده (از زمان آموزش).
- **Knowledge Source:** هر منبع بیرونی که اطلاعات اون رو به RAG می‌رسونیم.
- **External Context:** اطلاعات بازیابی‌شده از منبع که موقع سؤال به prompt اضافه می‌شه.

## ۲. چرا لازمه؟

| محدودیت LLM | راه‌حل با Knowledge Source |
|---|---|
| Knowledge Cutoff (دانش قدیمی) | داده‌ی تازه |
| Private Data (داده‌ی خصوصی) | داده‌ی داخلی شرکت |
| Verifiability (بدون منبع) | قابلیت ارجاع و citation |

## ۳. دسته‌بندی منابع

| بر اساس ساختار | مثال | چالش |
|---|---|---|
| Unstructured | PDF، Word، TXT | استخراج متن |
| Semi-structured | HTML، Markdown، JSON | حفظ سلسله‌مراتب |
| Structured | SQL، CSV، API | تبدیل به متن یا query مستقیم |

| بر اساس دسترسی | توضیح | مثال |
|---|---|---|
| Pre-indexed | قبلاً ایندکس شده | آیین‌نامه، FAQ |
| Live | موقع سؤال از API/DB | مانده‌ی مرخصی، وضعیت تیکت |
| Hybrid | ترکیب هر دو | اکثر سیستم‌های واقعی |

> **نکته:** داده‌ی عددی لحظه‌ای (مثل مانده‌ی مرخصی) باید با query دقیق یا tool جواب داده بشه، نه vector search.

## ۴. ویژگی‌های منبع دانش خوب

1. **Authoritative:** رسمی و معتبر
2. **Fresh:** دارای نسخه و تاریخ
3. **Owned:** مالک مشخص
4. **Scoped:** سطح دسترسی مشخص
5. **Traceable:** قابل ارجاع به منبع اصلی

## ۵. تعارض بین منابع

وقتی دو نسخه از یک سند هم‌زمان ایندکس باشن، retriever هر دو رو میاره و LLM گیج می‌شه.

**راه‌حل‌ها:**
- آرشیو یا حذف نسخه‌های قدیمی از ایندکس فعال
- فیلتر `status=active` از طریق metadata
- ذخیره‌ی `effective_date` و `version` برای هر chunk
- دستور در prompt: «در صورت تعارض، جدیدترین تاریخ رو ترجیح بده و تعارض رو اعلام کن»

## ۶. فرمت تزریق Context به prompt

```
[منبع 1 | leave_policy.pdf | صفحه 4 | نسخه 2025]
متن chunk ...
```

این برچسب‌ها: امکان citation، تشخیص منبع، و حل تعارض را فراهم می‌کنن.

## ۷. نقشه‌ی منابع پروژه‌ی HR

| منبع | فرمت | دسترسی | سطح |
|---|---|---|---|
| آیین‌نامه‌ی مرخصی | PDF | Pre-indexed | عمومی |
| سؤالات متداول | Markdown | Pre-indexed | عمومی |
| صفحات اینترانت | HTML | Pre-indexed | عمومی |
| سیاست حقوق مدیران | PDF | Pre-indexed | مدیران |
| مانده‌ی مرخصی کارمند | دیتابیس | Live (tool) | شخصی |

## ۸. ایده‌ی کد این نوت‌بوک

- **Source Registry:** ثبت مرکزی منابع با owner، version، status و access_level
- **ingestion انتخابی:** فقط منابع `active` ایندکس می‌شن
- **Metadata Inheritance:** هر chunk متادیتای منبعش رو به ارث می‌بره
- **تعارض نسخه‌ها:** نمایش اینکه بدون فیلتر `status` چه اتفاقی می‌افته و با فیلتر چطور حل می‌شه
- **Router ساده:** تشخیص اینکه سؤال باید از ایندکس (RAG) جواب داده بشه یا از دیتابیس زنده
- **فرمت‌کردن Context:** ساخت context با برچسب منبع، صفحه و نسخه

## ۹. چک‌لیست production

- [ ] هر منبع در Source Registry ثبت شده (owner، version، access_level، status)
- [ ] فقط منابع `active` ایندکس می‌شن
- [ ] نسخه‌های قدیمی آرشیو یا حذف می‌شن
- [ ] هر chunk متادیتای منبعش رو به ارث می‌بره
- [ ] داده‌ی لحظه‌ای و عددی از طریق Live tool جواب داده می‌شه، نه vector search
- [ ] Context با برچسب منبع، صفحه و نسخه به LLM داده می‌شه
- [ ] تعارض بین منابع در prompt مدیریت می‌شه

In [1]:
import re
import zlib
from dataclasses import dataclass, field
from datetime import date

import numpy as np


# ======================================================================
# 1) Source Registry: ثبت مرکزی منابع دانش
# ======================================================================
@dataclass
class KnowledgeSource:
    source_id: str
    path: str
    fmt: str                 # pdf | md | html | txt
    owner: str
    access_level: str        # public | management | hr
    version: str
    effective_date: date
    status: str = "active"   # active | archived
    content: str = ""        # برای دمو: متن سند (در واقعیت از فایل خونده می‌شه)


registry = [
    KnowledgeSource(
        "leave_2025", "leave_policy_2025.pdf", "pdf", "hr-team", "public",
        "2025", date(2025, 3, 1), "active",
        "مرخصی استحقاقی هر کارمند ۳۰ روز در سال است. مرخصی بدون حقوق حداکثر ۳۰ روز در سال است.",
    ),
    KnowledgeSource(
        "leave_2023", "leave_policy_2023.pdf", "pdf", "hr-team", "public",
        "2023", date(2023, 3, 1), "archived",
        "مرخصی استحقاقی هر کارمند ۲۶ روز در سال است.",
    ),
    KnowledgeSource(
        "insurance", "insurance.md", "md", "hr-team", "public",
        "2025", date(2025, 1, 10), "active",
        "بیمه تکمیلی هزینه‌های درمان، دندانپزشکی و بستری را پوشش می‌دهد.",
    ),
    KnowledgeSource(
        "salary_mgmt", "salary_policy.pdf", "pdf", "finance", "management",
        "2025", date(2025, 2, 1), "active",
        "جدول حقوق و مزایای مدیران ارشد محرمانه است.",
    ),
]


# ======================================================================
# 2) Embedder و Store ساده (همان الگوی نوت‌بوک قبلی)
# ======================================================================
class ToyEmbedder:
    STOPWORDS = {"و", "در", "به", "از", "که", "را", "با", "این", "است", "هر", "می", "ده", "چی", "چند", "روز", "چنده"}

    def __init__(self, dim=256):
        self.dim = dim

    def _embed(self, text):
        vec = np.zeros(self.dim, dtype=np.float32)
        for tok in re.findall(r"\w+", text.lower()):
            if tok not in self.STOPWORDS:
                vec[zlib.crc32(tok.encode()) % self.dim] += 1.0
        n = np.linalg.norm(vec)
        return vec / n if n > 0 else vec

    def embed_documents(self, texts):
        return [self._embed(t) for t in texts]

    def embed_query(self, text):
        return self._embed(text)


@dataclass
class Chunk:
    id: str
    text: str
    metadata: dict = field(default_factory=dict)
    score: float = 0.0


class InMemoryStore:
    def __init__(self):
        self.chunks, self.matrix = [], None

    def upsert(self, chunks, vectors):
        self.chunks += chunks
        new = np.vstack(vectors)
        self.matrix = new if self.matrix is None else np.vstack([self.matrix, new])

    def search(self, q_vec, k=5, filters=None):
        filters = filters or {}
        scores = self.matrix @ q_vec
        hits = []
        for i in np.argsort(-scores):
            if scores[i] <= 0:
                break
            c = self.chunks[i]
            if any(c.metadata.get(key) not in allowed for key, allowed in filters.items()):
                continue
            hits.append(Chunk(c.id, c.text, c.metadata, float(scores[i])))
            if len(hits) == k:
                break
        return hits


# ======================================================================
# 3) Ingestion: Metadata Inheritance + (اختیاری) فقط منابع active
# ======================================================================
def build_index(registry, embedder, only_active=True):
    store = InMemoryStore()
    sources = [s for s in registry if (s.status == "active" or not only_active)]
    chunks = [
        Chunk(
            id=s.source_id,
            text=s.content,
            metadata={
                "source": s.path,
                "owner": s.owner,
                "version": s.version,
                "effective_date": s.effective_date.isoformat(),
                "status": s.status,
                "dept": s.access_level,
            },
        )
        for s in sources
    ]
    store.upsert(chunks, embedder.embed_documents([c.text for c in chunks]))
    return store, len(chunks)


# ======================================================================
# 4) Router ساده: RAG یا Live Tool؟
# ======================================================================
LIVE_PATTERNS = [r"مانده", r"چند روز.*(باقی|مانده)", r"وضعیت تیکت", r"موجودی"]

def route(question: str) -> str:
    """اگر سؤال درباره‌ی داده‌ی شخصی/لحظه‌ای باشه → live، وگرنه → rag"""
    return "live" if any(re.search(p, question) for p in LIVE_PATTERNS) else "rag"

def live_leave_balance_tool(employee_id: str) -> str:
    fake_db = {"E100": 12, "E200": 5}   # در واقعیت: query روی دیتابیس HR
    return f"مانده‌ی مرخصی شما {fake_db.get(employee_id, 0)} روز است."


# ======================================================================
# 5) فرمت‌کردن External Context با برچسب منبع
# ======================================================================
def format_context(hits):
    blocks = []
    for i, h in enumerate(hits, 1):
        m = h.metadata
        header = f"[منبع {i} | {m['source']} | نسخه {m['version']} | تاریخ {m['effective_date']}]"
        blocks.append(f"{header}\n{h.text}")
    return "\n\n".join(blocks)


# ======================================================================
# 6) اجرا
# ======================================================================
embedder = ToyEmbedder()
question = "مرخصی استحقاقی چند روزه؟"

print("=" * 60, "\nالف) بدون مدیریت نسخه: همه‌ی منابع ایندکس می‌شن\n" + "=" * 60)
store_all, n = build_index(registry, embedder, only_active=False)
print(f"تعداد chunk ایندکس‌شده: {n}")
for h in store_all.search(embedder.embed_query(question), k=3, filters={"dept": ["public"]}):
    print(f"  {h.score:.2f} | {h.metadata['source']} (v{h.metadata['version']}, {h.metadata['status']}): {h.text}")
print("⚠️ دو جواب متناقض (۲۶ و ۳۰ روز) برگشت!")

print("\n" + "=" * 60, "\nب) فقط منابع active ایندکس می‌شن\n" + "=" * 60)
store, n = build_index(registry, embedder, only_active=True)
print(f"تعداد chunk ایندکس‌شده: {n}")
hits = store.search(embedder.embed_query(question), k=3, filters={"dept": ["public"]})
for h in hits:
    print(f"  {h.score:.2f} | {h.metadata['source']} (v{h.metadata['version']}): {h.text}")
print("✅ فقط نسخه‌ی معتبر برگشت.")

print("\n" + "=" * 60, "\nج) فیلتر status به‌جای حذف از ایندکس\n" + "=" * 60)
hits = store_all.search(embedder.embed_query(question), k=3,
                        filters={"dept": ["public"], "status": ["active"]})
for h in hits:
    print(f"  {h.score:.2f} | {h.metadata['source']} (v{h.metadata['version']}): {h.text}")

print("\n" + "=" * 60, "\nد) Context فرمت‌شده با برچسب منبع\n" + "=" * 60)
print(format_context(hits))

print("\n" + "=" * 60, "\nهـ) Router: RAG یا Live؟\n" + "=" * 60)
for q in ["مرخصی استحقاقی چند روزه؟", "مانده‌ی مرخصی من چنده؟", "بیمه تکمیلی چی پوشش می‌ده؟"]:
    r = route(q)
    print(f"[{r.upper():4}] {q}")
    if r == "live":
        print("       →", live_leave_balance_tool("E100"))

الف) بدون مدیریت نسخه: همه‌ی منابع ایندکس می‌شن
تعداد chunk ایندکس‌شده: 4
  0.52 | leave_policy_2023.pdf (v2023, archived): مرخصی استحقاقی هر کارمند ۲۶ روز در سال است.
  0.50 | leave_policy_2025.pdf (v2025, active): مرخصی استحقاقی هر کارمند ۳۰ روز در سال است. مرخصی بدون حقوق حداکثر ۳۰ روز در سال است.
  0.19 | insurance.md (v2025, active): بیمه تکمیلی هزینه‌های درمان، دندانپزشکی و بستری را پوشش می‌دهد.
⚠️ دو جواب متناقض (۲۶ و ۳۰ روز) برگشت!

ب) فقط منابع active ایندکس می‌شن
تعداد chunk ایندکس‌شده: 3
  0.50 | leave_policy_2025.pdf (v2025): مرخصی استحقاقی هر کارمند ۳۰ روز در سال است. مرخصی بدون حقوق حداکثر ۳۰ روز در سال است.
  0.19 | insurance.md (v2025): بیمه تکمیلی هزینه‌های درمان، دندانپزشکی و بستری را پوشش می‌دهد.
✅ فقط نسخه‌ی معتبر برگشت.

ج) فیلتر status به‌جای حذف از ایندکس
  0.50 | leave_policy_2025.pdf (v2025): مرخصی استحقاقی هر کارمند ۳۰ روز در سال است. مرخصی بدون حقوق حداکثر ۳۰ روز در سال است.
  0.19 | insurance.md (v2025): بیمه تکمیلی هزینه‌های درمان، دندانپزشکی و بستری را پوش